# LTO example

In [91]:
%%writefile modulo.f90
! modulo.f90
MODULE utilidades
  IMPLICIT NONE
CONTAINS
  FUNCTION dobrar(x) RESULT(y)
    IMPLICIT NONE
    INTEGER, INTENT(IN) :: x
    INTEGER             :: y
    y = x * 2
    y = y + 1
    if (x > 0) then
      y = 0
    endif
  END FUNCTION dobrar
END MODULE utilidades

Overwriting modulo.f90


In [92]:
%%writefile principal.f90
! principal.f90
PROGRAM meu_programa_lto
  USE utilidades
  IMPLICIT NONE

  integer :: valor_inicial
  integer :: resultado

  WRITE(*,*) "Entre com um valor numerico:"
  READ(*,*) valor_inicial

  resultado = dobrar(valor_inicial)

  PRINT *, "O resultado final e: ", resultado

END PROGRAM meu_programa_lto

Overwriting principal.f90


In [93]:
! gfortran modulo.f90 principal.f90 -o meu_executavel

In [94]:
! objdump -d -j .text meu_executavel | tail -n +78

00000000000011b9 <__utilidades_MOD_dobrar>:
    11b9:	55                   	push   %rbp
    11ba:	48 89 e5             	mov    %rsp,%rbp
    11bd:	48 89 7d e8          	mov    %rdi,-0x18(%rbp)
    11c1:	48 8b 45 e8          	mov    -0x18(%rbp),%rax
    11c5:	8b 00                	mov    (%rax),%eax
    11c7:	01 c0                	add    %eax,%eax
    11c9:	89 45 fc             	mov    %eax,-0x4(%rbp)
    11cc:	83 45 fc 01          	addl   $0x1,-0x4(%rbp)
    11d0:	48 8b 45 e8          	mov    -0x18(%rbp),%rax
    11d4:	8b 00                	mov    (%rax),%eax
    11d6:	85 c0                	test   %eax,%eax
    11d8:	7e 07                	jle    11e1 <__utilidades_MOD_dobrar+0x28>
    11da:	c7 45 fc 00 00 00 00 	movl   $0x0,-0x4(%rbp)
    11e1:	8b 45 fc             	mov    -0x4(%rbp),%eax
    11e4:	5d                   	pop    %rbp
    11e5:	c3                   	ret    

00000000000011e6 <MAIN__>:
    11e6:	55                   	push   %rbp
    11e7:	48 89 e5             	mov    %rsp,

In [95]:
! gfortran -O3 -flto modulo.f90 principal.f90 -o meu_executavel_lto

In [96]:
! objdump -d -j .text meu_executavel_lto


meu_executavel_lto:     file format elf64-x86-64


Disassembly of section .text:

00000000000010d0 <main>:
    10d0:	48 83 ec 08          	sub    $0x8,%rsp
    10d4:	e8 c7 ff ff ff       	call   10a0 <_gfortran_set_args@plt>
    10d9:	48 8d 35 70 0f 00 00 	lea    0xf70(%rip),%rsi        # 2050 <options.3.0>
    10e0:	bf 07 00 00 00       	mov    $0x7,%edi
    10e5:	e8 a6 ff ff ff       	call   1090 <_gfortran_set_options@plt>
    10ea:	e8 01 01 00 00       	call   11f0 <MAIN__>
    10ef:	31 c0                	xor    %eax,%eax
    10f1:	48 83 c4 08          	add    $0x8,%rsp
    10f5:	c3                   	ret    
    10f6:	66 2e 0f 1f 84 00 00 	cs nopw 0x0(%rax,%rax,1)
    10fd:	00 00 00 

0000000000001100 <_start>:
    1100:	f3 0f 1e fa          	endbr64 
    1104:	31 ed                	xor    %ebp,%ebp
    1106:	49 89 d1             	mov    %rdx,%r9
    1109:	5e                   	pop    %rsi
    110a:	48 89 e2             	mov    %rsp,%rdx
    110d:	48 83 e4 f0          	and    $0x